In [1]:
import time
from functools import wraps

def ttl_cache(seconds):

    def decorator(func):
        cache = {}
        stats = {"hits": 0, "misses": 0}

        @wraps(func)
        def wrapper(*args, **kwargs):

            key = (args, tuple(sorted(kwargs.items())))
            now = time.monotonic()

            try:
                if key in cache:
                    result, expiry = cache[key]

                    if now < expiry:
                        stats["hits"] += 1
                        return result

                stats["misses"] += 1
                result = func(*args, **kwargs)
                cache[key] = (result, now + seconds)

                return result

            except TypeError:
                stats["misses"] += 1
                return func(*args, **kwargs)

        wrapper.stats = stats
        return wrapper

    return decorator


@ttl_cache(5)
def add(x, y):
    print("Calculating...")
    return x + y


print(add(x=2, y=3))
print(add(y=3, x=2))
print(add.stats)

Calculating...
5
5
{'hits': 1, 'misses': 1}
